# Spatial VLM — Training Notebook (Colab)

运行环境：Google Colab（GPU 选 A100 或 T4）

In [ ]:
# ── 2. 安装依赖 ────────────────────────────────────────────────
# Colab 已预装 torch，只需要补装 transformers / peft / accelerate
os.system("pip install -q transformers peft accelerate")

In [ ]:
# ── 3. 验证环境 ────────────────────────────────────────────────
import sys
import torch

# 把仓库根目录加入 path，让 spatial_encoder 包可以被导入
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("torch version :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("显存          :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("⚠️  未检测到 GPU，请在 Colab 菜单 Runtime → Change runtime type 里选 GPU")

## Step 3 — 挂载数据 + 构建训练样本

`.blend` 和图片太大不能放 GitHub，需要从 Google Drive 读取。  
请提前把 `sample/` 文件夹上传到你的 Google Drive。

In [ ]:
# ── 4. 挂载 Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# 修改为你 Drive 里 sample/ 的实际路径
DRIVE_SAMPLE = "/content/drive/MyDrive/blender-adapter/sample"

SCENE_JSON  = f"{DRIVE_SAMPLE}/ffc5bced-d6c3-4475-a289-0da7ec342be7-Brick_house_Glass_wall_info.json"
SCENE_IMAGE = f"{DRIVE_SAMPLE}/imagefolder/ffc5bced-d6c3-4475-a289-0da7ec342be7-Brick_house_Glass_wall.jpg"

import os
assert os.path.exists(SCENE_JSON),  f"找不到 JSON：{SCENE_JSON}"
assert os.path.exists(SCENE_IMAGE), f"找不到图片：{SCENE_IMAGE}"
print("数据文件确认存在 ✓")

In [ ]:
import json
import numpy as np
from spatial_encoder.qa_generator import generate_qa_samples
from spatial_encoder.token_injection import build_sample

# SCENE_JSON / SCENE_IMAGE 已在上面 cell 定义
with open(SCENE_JSON) as f:
    scene = json.load(f)

qa_samples = generate_qa_samples(scene, n_direction=5, n_compare=5, n_nearest=2, n_count=1)

built = []
for qa in qa_samples:
    if qa.object_names:
        s = build_sample(
            scene_json=scene,
            question=qa.question,
            answer=qa.answer,
            object_names=qa.object_names,
        )
        s["image_path"] = SCENE_IMAGE
        s["qa_type"] = qa.qa_type
        built.append(s)

print(f"built {len(built)} samples")
for s in built[:3]:
    print(f"\n[{s['qa_type']}]")
    print("  Q:", s["prompt"])
    print("  A:", s["answer"])
    print("  spatial_vectors shape:", s["spatial_vectors"].shape)

## Step 4 — 加载模型

首次运行会从 HuggingFace 下载 Qwen2-VL-7B 权重（约 15GB）。  
显存不够时改用 `Qwen/Qwen2-VL-2B-Instruct`（约 5GB）。

```bash
huggingface-cli login  # 需要先在终端登录
```

In [ ]:
from spatial_encoder.token_injection import SpatialQwen2VL

MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct"
# MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # 显存不够时用这个

model = SpatialQwen2VL(MODEL_NAME)
tokenizer = model.tokenizer

print("模型加载完成")
print("hidden_size    :", model.vlm.config.hidden_size)
print("position_encoder 参数量:", sum(p.numel() for p in model.position_encoder.parameters()))

## Step 4b — 单条推理（验证 token 注入正确）

在开始训练前，先用一条样本做 forward，确认没有 shape 报错、没有 NaN。

In [ ]:
from PIL import Image

sample = built[0]

# Tokenize prompt（含 <obj_N> 占位符）
inputs = tokenizer(
    sample["prompt"],
    return_tensors="pt",
    padding=True,
).to("cuda")

# 图像
image = Image.open(sample["image_path"]).convert("RGB")
# Qwen2-VL 的 processor 处理图像；这里用简化方式先验证文本+坐标通路
# 完整多模态见 Step 5

# spatial_vectors: (1, N_obj, 6)
spatial = torch.tensor(sample["spatial_vectors"], dtype=torch.float32) \
              .unsqueeze(0).to("cuda")

print("input_ids shape    :", inputs["input_ids"].shape)
print("spatial_vectors    :", spatial.shape)

# Forward（不含图像，先验证 token 注入通路）
with torch.no_grad():
    out = model(
        input_ids=inputs["input_ids"],
        spatial_vectors=spatial,
    )
print("logits shape       :", out.logits.shape)
print("logits has NaN     :", torch.isnan(out.logits).any().item())
print("✓ token 注入 forward 正常")

In [ ]:
# 生成一条回答，看模型在没有训练时的 baseline 输出
output_ids = model.generate(
    input_ids=inputs["input_ids"],
    spatial_vectors=spatial,
    max_new_tokens=64,
    do_sample=False,
)
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Prompt :", sample["prompt"])
print("GT     :", sample["answer"])
print("Model  :", response)

## Step 5 — 添加 LoRA

只训练 LoRA 层 + 位置编码器，Qwen2-VL 主体权重全程冻结。

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
)
model.vlm = get_peft_model(model.vlm, lora_config)
model.vlm.print_trainable_parameters()

# 位置编码器也要训练
for p in model.position_encoder.parameters():
    p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总可训练参数：{trainable / 1e6:.1f}M")

## Step 6 — 训练循环

`labels` 里 question 部分设为 `-100`（不计算 loss），只对 answer 部分计算 cross-entropy。

In [ ]:
def make_labels(input_ids: torch.Tensor, answer: str, tokenizer) -> torch.Tensor:
    """
    input_ids: (1, L)  prompt + answer 拼在一起后的 token ids
    返回 labels: 同 shape，question 部分为 -100，answer 部分保留
    """
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_len = len(answer_ids)

    labels = input_ids.clone()
    labels[0, :-answer_len] = -100  # mask 掉 question 部分
    return labels


def collate_batch(samples: list[dict], tokenizer, device="cuda"):
    """把一个 list of built samples 拼成一个 batch。"""
    full_texts = [s["prompt"] + s["answer"] for s in samples]
    inputs = tokenizer(full_texts, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(device)

    labels_list = []
    for i, s in enumerate(samples):
        lab = make_labels(inputs["input_ids"][i:i+1], s["answer"], tokenizer)
        labels_list.append(lab)
    labels = torch.cat(labels_list, dim=0)

    # spatial_vectors: pad 到同一 N_obj（取 batch 内最大值）
    max_objs = max(s["spatial_vectors"].shape[0] for s in samples)
    sv_list = []
    for s in samples:
        sv = s["spatial_vectors"]          # (N, 6)
        pad = np.zeros((max_objs - sv.shape[0], 6), dtype=np.float32)
        sv_list.append(np.concatenate([sv, pad], axis=0))
    spatial = torch.tensor(np.stack(sv_list), dtype=torch.float32).to(device)  # (B, max_N, 6)

    return inputs["input_ids"], labels, spatial

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

BATCH_SIZE = 4
NUM_EPOCHS = 3
LOG_EVERY   = 10

optimizer = AdamW([
    {"params": model.vlm.parameters(),             "lr": 2e-4},
    {"params": model.position_encoder.parameters(), "lr": 1e-3},
])
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * (len(built) // BATCH_SIZE + 1))

model.train()
model.position_encoder.train()

step = 0
for epoch in range(NUM_EPOCHS):
    # 简单 shuffle
    import random
    random.shuffle(built)

    for i in range(0, len(built), BATCH_SIZE):
        batch = built[i: i + BATCH_SIZE]
        if not batch:
            continue

        input_ids, labels, spatial = collate_batch(batch, tokenizer)

        outputs = model(
            input_ids=input_ids,
            spatial_vectors=spatial,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if step % LOG_EVERY == 0:
            print(f"epoch {epoch+1}  step {step:4d}  loss {loss.item():.4f}")
        step += 1

print("训练完成")

## Step 7 — 保存权重

只保存可训练部分（LoRA adapter + position encoder），不保存冻结的主体权重。

In [ ]:
import os

SAVE_DIR = "checkpoints/spatial_vlm"
os.makedirs(SAVE_DIR, exist_ok=True)

# LoRA adapter
model.vlm.save_pretrained(SAVE_DIR)

# Position encoder
torch.save(model.position_encoder.state_dict(),
           os.path.join(SAVE_DIR, "position_encoder.pt"))

print(f"权重已保存到 {SAVE_DIR}/")
print(os.listdir(SAVE_DIR))

## Step 8 — Evaluation

对每条测试样本：生成模型输出 → 解析答案 → 和 ground truth 比对。

In [ ]:
import re
from collections import defaultdict

DIRECTION_KEYWORDS = {
    "右侧": "RIGHT", "右前方": "RIGHT_FRONT",
    "左侧": "LEFT",  "左前方": "LEFT_FRONT",
    "正前方": "FRONT", "正后方": "BACK",
}

def parse_direction(text: str) -> str:
    for kw, label in DIRECTION_KEYWORDS.items():
        if kw in text:
            return label
    return "UNKNOWN"

def parse_choice(text: str, n_choices: int) -> int:
    """从输出里找 <obj_N> 或"第N个"，返回 index；找不到返回 -1。"""
    for i in range(n_choices):
        if f"<obj_{i}>" in text or f"obj_{i}" in text:
            return i
    m = re.search(r"第\s*(\d+)", text)
    if m:
        return int(m.group(1)) - 1
    return -1

def parse_count(text: str) -> int:
    m = re.search(r"(\d+)\s*个", text)
    return int(m.group(1)) if m else -1


def evaluate(model, tokenizer, test_samples: list[dict], device="cuda"):
    model.eval()
    results = defaultdict(lambda: {"correct": 0, "total": 0})

    for s in test_samples:
        # Tokenize
        enc = tokenizer(s["prompt"], return_tensors="pt").to(device)
        spatial = torch.tensor(s["spatial_vectors"], dtype=torch.float32) \
                      .unsqueeze(0).to(device)

        with torch.no_grad():
            out_ids = model.generate(
                input_ids=enc["input_ids"],
                spatial_vectors=spatial,
                max_new_tokens=64,
                do_sample=False,
            )
        pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)

        qt = s["qa_type"]
        results[qt]["total"] += 1

        if qt == "direction":
            gt  = parse_direction(s["answer"])
            pr  = parse_direction(pred)
            correct = (gt == pr and gt != "UNKNOWN")

        elif qt in ("distance_compare", "nearest"):
            n = len(s["object_names"])
            gt_idx = parse_choice(s["answer"], n)
            pr_idx = parse_choice(pred, n)
            correct = (gt_idx == pr_idx and gt_idx != -1)

        elif qt == "count":
            gt_n = parse_count(s["answer"])
            pr_n = parse_count(pred)
            correct = (gt_n == pr_n and gt_n != -1)
        else:
            correct = False

        if correct:
            results[qt]["correct"] += 1

    print(f"\n{'QA Type':<20} {'Acc':>8}  (correct / total)")
    print("-" * 45)
    total_c, total_t = 0, 0
    for qt, v in results.items():
        acc = v["correct"] / v["total"] if v["total"] else 0
        print(f"{qt:<20} {acc:>7.1%}  ({v['correct']} / {v['total']})")
        total_c += v["correct"]
        total_t += v["total"]
    overall = total_c / total_t if total_t else 0
    print("-" * 45)
    print(f"{'Overall':<20} {overall:>7.1%}  ({total_c} / {total_t})")
    return results


# 用 built 里后 20% 作为临时测试集（正式实验应用独立测试场景）
split = int(len(built) * 0.8)
test_samples = built[split:]
evaluate(model, tokenizer, test_samples)